In [ ]:
# CÉLULA 1: ANÁLISE DE RESULTADOS (UM BOXPLOT PARA CADA PISTA)
# =============================================================================
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# =============================================================================
# 1. CONFIGURAÇÕES DE BUSCA (Ajuste o Range Aqui)
# =============================================================================
pasta_base = r'C:\IC_alex\TreinamentoMGGP_2_0\RESULTADOS' 
prefixo_pasta = 'Treinamento_'

# ESCOLHA O RANGE DE PASTAS AQUI:
pasta_inicio = 1   
pasta_fim = 14


# =============================================================================
# 2. COLETANDO OS DADOS
# =============================================================================
dados_totais = []
print(f"🔍 Buscando relatórios da pasta {pasta_inicio} até {pasta_fim}...\n")

for i in range(pasta_inicio, pasta_fim + 1):
    nome_pasta = f"{prefixo_pasta}{i}"
    caminho_pasta = os.path.join(pasta_base, nome_pasta)
    
    if os.path.exists(caminho_pasta):
        arquivos_csv = [f for f in os.listdir(caminho_pasta) if f.endswith('.csv') and 'relatorio' in f.lower()]
        
        for arq in arquivos_csv:
            caminho_csv = os.path.join(caminho_pasta, arq)
            try:
                df = pd.read_csv(caminho_csv, sep=';', decimal=',')
                df['Origem'] = f"Treino {i}" 
                dados_totais.append(df)
                print(f"   ✅ Relatório carregado: {nome_pasta}\\{arq}")
            except Exception as e:
                print(f"   ⚠️ Erro ao ler {arq} na pasta {nome_pasta}: {e}")
    else:
        print(f"   ❌ Pasta não encontrada (Ignorando): {nome_pasta}")

# =============================================================================
# 3. PLOTANDO UM GRÁFICO SEPARADO PARA CADA PISTA
# =============================================================================
if len(dados_totais) > 0:
    df_final = pd.concat(dados_totais, ignore_index=True)
    
    # Descobre quais são as pistas (Wangs) únicas que existem nos CSVs
    pistas_unicas = df_final['Pista'].unique()
    
    print("\n" + "="*70)
    print("📈 GERANDO GRÁFICOS INDIVIDUAIS POR PISTA...")
    print("="*70)
    
    for pista in pistas_unicas:
        # Filtra os dados apenas para a pista atual do loop
        df_pista = df_final[df_final['Pista'] == pista]
        
        # Cria uma figura para esta pista com Vx na esquerda e Vy na direita
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle(f'Desempenho na Pista: {pista}', fontsize=18, fontweight='bold', y=1.05)
        
        # --- Boxplot Vx ---
        sns.boxplot(data=df_pista, x='Origem', y='MAPE_Vx (%)', ax=axes[0], palette='Blues', hue='Origem', legend=False)
        axes[0].set_title('Erro Vx (Velocidade Longitudinal)', fontsize=14)
        axes[0].set_ylabel('MAPE (%)', fontsize=12)
        axes[0].set_xlabel('Sessões de Treinamento', fontsize=12)
        axes[0].grid(True, axis='y', alpha=0.3)
        
        # --- Boxplot Vy ---
        sns.boxplot(data=df_pista, x='Origem', y='MAPE_Vy (%)', ax=axes[1], palette='Reds', hue='Origem', legend=False)
        axes[1].set_title('Erro Vy (Velocidade Lateral)', fontsize=14)
        axes[1].set_ylabel('MAPE (%)', fontsize=12)
        axes[1].set_xlabel('Sessões de Treinamento', fontsize=12)
        axes[1].grid(True, axis='y', alpha=0.3)
        
        plt.tight_layout()
        plt.show() # Mostra o gráfico desta pista e vai para a próxima

    # --- Tabela de Resumo ---
    print("\n" + "="*70)
    print("📊 RESUMO DAS MÉDIAS DE MAPE POR PISTA E TREINAMENTO")
    print("="*70)
    resumo = df_final.groupby(['Pista', 'Origem'])[['MAPE_Vx (%)', 'MAPE_Vy (%)']].mean().round(2)
    print(resumo.to_string())
    
else:
    print("\n⚠️ Nenhum dado encontrado no range selecionado para plotar o Boxplot.")

In [3]:
import pandas as pd
import os

# 1. Configuração dos Diretórios
diretorios_pcs = {
    "PC_1": r"C:\IC_alex\TreinamentoMGGP_2_0\RESULTADOS",
    # "PC_2": r"C:\Caminho\Do\PC2\RESULTADOS", 
    # "PC_3": r"E:\Caminho\Do\PC3\RESULTADOS"  
}

# 2. Configuração do Range de Treinamentos
treinamento_inicio = 1
treinamento_fim = 19  

dados_consolidados = []

print("Iniciando a varredura dos arquivos...\n")

# 3. Processamento dos Arquivos
for pc_nome, diretorio_base in diretorios_pcs.items():
    if not os.path.exists(diretorio_base):
        print(f"Aviso: Diretório base do {pc_nome} não encontrado: {diretorio_base}")
        continue
        
    for i in range(treinamento_inicio, treinamento_fim + 1):
        pasta_treino = f"Treinamento_{i}"
        
        # ATENÇÃO: Se o nome do arquivo mudou, ajuste este texto abaixo:
        nome_arquivo_csv = "relatorio_validacao_rmse.csv" 
        
        caminho_arquivo = os.path.join(diretorio_base, pasta_treino, nome_arquivo_csv)
        
        if os.path.exists(caminho_arquivo):
            try:
                df_temp = pd.read_csv(caminho_arquivo, sep=None, engine='python', header=None)
                df_temp = df_temp.iloc[:, :5]
                df_temp.columns = ['Modelo', 'Pista', 'RMSE_vx', 'RMSE_vy', 'RMSE_Total']
                df_temp = df_temp.dropna(subset=['Pista'])
                
                # Conversão e limpeza dos dados decimais
                for col in ['RMSE_vx', 'RMSE_vy', 'RMSE_Total']:
                    df_temp[col] = df_temp[col].astype(str).str.strip().str.replace(',', '.', regex=False)
                    df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
                
                df_temp = df_temp.dropna(subset=['RMSE_vx', 'RMSE_vy'])
                
                # Calcula a média das 10 rodadas de cada pista
                df_media_pistas = df_temp.groupby('Pista')[['RMSE_vx', 'RMSE_vy']].mean().reset_index()
                df_media_pistas['PC'] = pc_nome
                df_media_pistas['Treinamento'] = i
                
                dados_consolidados.append(df_media_pistas)
                print(f"Sucesso ao processar: {pc_nome} -> {pasta_treino}")
                
            except Exception as e:
                print(f"Erro ao ler o arquivo {caminho_arquivo}: {e}")
        else:
            print(f"Aviso: Arquivo não encontrado - {caminho_arquivo}")

# 4. Geração, Exibição e Salvamento Separado das Tabelas
if len(dados_consolidados) > 0:
    df_mestre = pd.concat(dados_consolidados, ignore_index=True)

    # Criando as matrizes Pivot separadas
    tabela_vx = df_mestre.pivot_table(index=['PC', 'Treinamento'], columns='Pista', values='RMSE_vx')
    tabela_vy = df_mestre.pivot_table(index=['PC', 'Treinamento'], columns='Pista', values='RMSE_vy')

    # --- MOSTRA AS DUAS TABELAS NO PROMPT ---
    print("\n" + "="*35 + " TABELA CONSOLIDADA RMSE Vx " + "="*35)
    print(tabela_vx)
    
    print("\n" + "="*35 + " TABELA CONSOLIDADA RMSE Vy " + "="*35)
    print(tabela_vy)
    print("="*98 + "\n")

    # --- SALVA EM DOIS ARQUIVOS EXCEL SEPARADOS NA MESMA PASTA DO SCRIPT ---
    arquivo_vx = "Tabela_RMSE_Vx.xlsx"
    arquivo_vy = "Tabela_RMSE_Vy.xlsx"
    
    tabela_vx.to_excel(arquivo_vx)
    tabela_vy.to_excel(arquivo_vy)

    print("Sucesso! Foram gerados dois arquivos na sua pasta:")
    print(f"1. Tabela de Vx salva em: {arquivo_vx}")
    print(f"2. Tabela de Vy salva em: {arquivo_vy}")
else:
    print("\nErro: Nenhum arquivo pôde ser processado.")

Iniciando a varredura dos arquivos...

Sucesso ao processar: PC_1 -> Treinamento_1
Sucesso ao processar: PC_1 -> Treinamento_2
Sucesso ao processar: PC_1 -> Treinamento_3
Sucesso ao processar: PC_1 -> Treinamento_4
Sucesso ao processar: PC_1 -> Treinamento_5
Erro ao ler o arquivo C:\IC_alex\TreinamentoMGGP_2_0\RESULTADOS\Treinamento_6\relatorio_validacao_rmse.csv: Length mismatch: Expected axis has 2 elements, new values have 5 elements
Sucesso ao processar: PC_1 -> Treinamento_7
Sucesso ao processar: PC_1 -> Treinamento_8
Sucesso ao processar: PC_1 -> Treinamento_9
Sucesso ao processar: PC_1 -> Treinamento_10
Sucesso ao processar: PC_1 -> Treinamento_11
Sucesso ao processar: PC_1 -> Treinamento_12
Sucesso ao processar: PC_1 -> Treinamento_13
Sucesso ao processar: PC_1 -> Treinamento_14
Sucesso ao processar: PC_1 -> Treinamento_15
Sucesso ao processar: PC_1 -> Treinamento_16
Sucesso ao processar: PC_1 -> Treinamento_17
Sucesso ao processar: PC_1 -> Treinamento_18
Sucesso ao processar: 